# Pure cancellation: two independent constructions

Compare maximum-of-lifetimes and sum-of-holding-times samplers with exact moments. Unit orders, no replenishment or market consumption.

Synthetic data only. All settings and random seeds are specified below.

In [1]:
from pathlib import Path
import sys, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
OUT = ROOT / "results" / "synthetic" / "queues"
OUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": .2})
print({"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__})


{'python': '3.12.0', 'numpy': '1.26.0', 'pandas': '2.1.1'}


In [2]:
from queue_models.cancellation import CancellationConfig, simulate, assess
rows=[]
for q in (1, 10, 100):
    for method in ("maximum", "stages"):
        sample=simulate(CancellationConfig(q,.2),samples=10000,seed=100+q,method=method)
        rows.append(dict(method=method, **assess(CancellationConfig(q,.2),sample)))
table=pd.DataFrame(rows)
display(table[["q","method","mean","theory_mean","mean_se","mean_z","cdf_pass"]])
table.to_csv(OUT/"cancellation.csv",index=False)
for method,g in table.groupby("method"):
    plt.errorbar(g.q,g["mean"],yerr=2*g.mean_se,marker="o",label=method)
g=table[table.method=="stages"]
plt.plot(g.q,g.theory_mean,"k--",label="Exact mean")
plt.xscale("log");plt.xlabel("Initial queue size");plt.ylabel("Depletion time (model units)");plt.legend()
plt.tight_layout();plt.savefig(OUT/"cancellation.png",dpi=140);plt.show()


,q,method,mean,theory_mean,mean_se,mean_z,cdf_pass
0,1,maximum,5.034493,5.000000,0.050000,0.689870,True
1,1,stages,5.034493,5.000000,0.050000,0.689870,True
2,10,maximum,14.703645,14.644841,0.062245,0.944709,True
3,10,stages,14.746103,14.644841,0.062245,1.626824,True
4,100,maximum,25.930815,25.936888,0.063933,-0.094982,True
5,100,stages,25.857983,25.936888,0.063933,-1.234165,True


/var/folders/p2/_bpfxjzn1l301l3jgtn6bgx80000gn/T/ipykernel_57265/2251022342.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout();plt.savefig(OUT/"cancellation.png",dpi=140);plt.show()


## Interpretation

Compare the displayed model reference with the simulation. Finite-sample agreement is an implementation check, not evidence of real-market calibration. See [assumptions](../../../docs/model_assumptions.md) and [limitations](../../../docs/validation_and_limitations.md).